In [17]:
import pathlib
import os
import datetime

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import torch
from torch.utils.tensorboard.writer import SummaryWriter
import trimesh

import utils, dataset, train
from encoder import *
from decoder import *

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
# Model name
current_time = datetime.datetime.now().strftime("%b%d_%H-%M")
denoiser_name = "1_Objects_100epochs_32batch_KL0.0001"
experiment_name = f"{current_time}_{denoiser_name}"


config = {
    'experiment_name': experiment_name,
    'device': 'cuda:0',
    'batch_size': 32,
    'resume_ckpt': None,
    'learning_rate': 0.0004,
    'step_size': 10, # scheduler step, one step is one batch
    'gamma': 1,
    'max_epochs': 100,
    'timesteps': 1000,
    'print_every_n': 15, # every n batches
    # 'validate_every_n': 10,
    # 'print_EMD_every_n': 1
}

model_config = {
    'last_epoch': 0,
}

In [19]:
# declare device
if torch.cuda.is_available() and config['device'].startswith('cuda'):
    device = torch.device(config['device'])
    print('Using device:', config['device'])
else:
    device = torch.device('cpu')
    print('Using CPU')

# create dataloaders
trainset = dataset.Dataset('overfit', config['timesteps'])
trainloader = torch.utils.data.DataLoader(trainset, batch_size=config['batch_size'], shuffle=True, num_workers=0, pin_memory=False)
# valset = dataset.Dataset('val', config['timesteps'])
# valloader = torch.utils.data.DataLoader(valset, batch_size=config['batch_size'], shuffle=False, num_workers=0)

encoder = Encoder(number_points=2048, in_channels=3, hidden_channels=128, latent_dim=64, clamp=False)
decoder = Decoder(number_points=2048, point_dim=3, hidden_dim=128, latent_dim=64, timesteps=1000, beta_start=1e-4, beta_end=0.02)

# move model to specified device
encoder.to(device)
decoder.to(device)
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=config['learning_rate'])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, config['step_size'], config['gamma'])
# scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.0006, steps_per_epoch=len(trainloader), epochs=config['max_epochs'], anneal_strategy='cos', three_phase=True)



# load model if resuming from checkpoint
# experiment_name = "May31_17-04_10_Objects_200epochs_32batch"
# utils.reload_model(encoder, decoder, optimizer, None, experiment_name, 'epoch136', device)


total_enc, trainable_enc = utils.count_parameters(encoder)
total_dec, trainable_dec = utils.count_parameters(decoder)
print(f"Encoder Total: {total_enc:,} | Trainable: {trainable_enc:,} | Model size: {utils.model_memory_size(encoder):.3f} MB")
print(f"Decoder Total: {total_dec:,} | Trainable: {trainable_dec:,} | Model size: {utils.model_memory_size(decoder):.3f} MB")

Using device: cuda:0
Encoder Total: 512,384 | Trainable: 512,384 | Model size: 1.955 MB
Decoder Total: 502,936 | Trainable: 502,936 | Model size: 1.938 MB


In [16]:
torch.cuda.empty_cache()

In [15]:
for i in range(27):
    print(min(0.01, 0.0001 * 1.2 ** i))

0.0001
0.00012
0.000144
0.00017279999999999997
0.00020736
0.00024883199999999994
0.00029859839999999994
0.00035831807999999995
0.0004299816959999999
0.0005159780351999999
0.0006191736422399998
0.0007430083706879997
0.0008916100448255996
0.0010699320537907195
0.0012839184645488633
0.001540702157458636
0.0018488425889503632
0.0022186111067404356
0.0026623333280885227
0.003194799993706227
0.0038337599924474727
0.004600511990936967
0.00552061438912436
0.006624737266949231
0.007949684720339077
0.009539621664406894
0.01


In [20]:
# start training
torch.cuda.empty_cache()
#tensorboard --logdir=diffusion_autoencoder/logs
# config['max_epochs'] = 200
# Create tensorboard writer    
log_path = pathlib.Path(f"logs/{datetime.datetime.now().strftime('%b%d')}/{config['experiment_name']}")
writer = SummaryWriter(log_path)

with open("debug.txt", "w", encoding="utf-8") as debug_file:
    train.train(encoder, decoder, trainloader, None, device, optimizer, scheduler, config, model_config, writer, None, debug_file)

writer.close()


[00/014] train_loss: 1.255
[00/029] train_loss: 1.122
Best Model:   1.1833502016961575
[01/012] train_loss: 1.061
[01/027] train_loss: 0.765
Best Model:   0.8387475507333875
[02/010] train_loss: 0.321
[02/025] train_loss: 0.205
Best Model:   0.22976198559626937
[03/008] train_loss: 0.159
[03/023] train_loss: 0.181
Best Model:   0.16498883964959532
[04/006] train_loss: 0.168
[04/021] train_loss: 0.192
[05/004] train_loss: 0.197
[05/019] train_loss: 0.164
Best Model:   0.16052889521233737
[06/002] train_loss: 0.162
[06/017] train_loss: 0.164
[07/000] train_loss: 0.182
[07/015] train_loss: 0.192
[07/030] train_loss: 0.174
[08/013] train_loss: 0.163
[08/028] train_loss: 0.182
[09/011] train_loss: 0.166
[09/026] train_loss: 0.166
[10/009] train_loss: 0.165
[10/024] train_loss: 0.175
[11/007] train_loss: 0.185
[11/022] train_loss: 0.188
[12/005] train_loss: 0.178
[12/020] train_loss: 0.161
[13/003] train_loss: 0.160
[13/018] train_loss: 0.155
Best Model:   0.15772420971188694
[14/001] train_

KeyboardInterrupt: 

In [25]:
#Load a model:
experiment_name = "Jun01_23-07_1_Objects_100epochs_32batch_KL0.0001"
utils.reload_model(encoder, decoder, None, None, experiment_name, 'best', device)

({'experiment_name': 'Jun01_23-07_1_Objects_100epochs_32batch_KL0.0001',
  'device': 'cuda:0',
  'batch_size': 32,
  'resume_ckpt': None,
  'learning_rate': 0.0004,
  'step_size': 10,
  'gamma': 1,
  'max_epochs': 100,
  'timesteps': 1000,
  'print_every_n': 15},
 {'last_epoch': 0})

In [26]:
generateDDIM = 1
generateDDPM = 1
number_of_points = 2048
DDIM_steps = 60
number_of_DDIM_iterations = 2
save = False
start = 20

if generateDDIM:
    for i in range(start, start + number_of_DDIM_iterations):
        object_index = 0
        mean, log_variance = encoder(trainset[object_index].unsqueeze(0))
        code = encoder.sample_latent_z(mean, log_variance)
        generated_pc_ddim = sample_ddim(decoder, code, n_points=number_of_points, steps=DDIM_steps)
        generated_pcd_ddim = trimesh.PointCloud(generated_pc_ddim.squeeze().cpu().numpy())
        utils.visualize_comparison(trainset[object_index], generated_pc_ddim, window_name="DDIM Target (Red) vs Generated (Blue)")
        if save:
            path = pathlib.Path(f"output/{experiment_name}")
            path.mkdir(parents=True, exist_ok=True)
            generated_pcd_ddim.export(path / f"ddim{DDIM_steps}_{number_of_points}_{i}.obj")

if generateDDPM:
        object_index = 0
        mean, log_variance = encoder(trainset[object_index].unsqueeze(0))
        code = encoder.sample_latent_z(mean, log_variance)
        generated_pc_ddim = sample_ddpm(decoder, code, n_points=number_of_points)
        generated_pcd_ddim = trimesh.PointCloud(generated_pc_ddim.squeeze().cpu().numpy())
        utils.visualize_comparison(trainset[object_index], generated_pc_ddim, window_name="DDIM Target (Red) vs Generated (Blue)")

Visualizing: Target is RED, Generated is BLUE.
Visualizing: Target is RED, Generated is BLUE.
Visualizing: Target is RED, Generated is BLUE.


In [10]:
#VAE latent space visualization
import torch
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# 1. Extract latent representations
encoder.eval()
all_latents = []
all_labels = []

testset = dataset.Dataset('test', config['timesteps'])

with torch.no_grad():
    for index in range(len(testset)):
        data = testset[index]
        data = data.to(device)
        print(f"Input shape before encoder: {data.shape}")
        print(f"Index: {index}")
        # Assuming your encoder returns mu and logvar
        mu, _ = encoder(data.unsqueeze(0)) 
        all_latents.append(mu.cpu().numpy())
        all_labels.append(index)

latents = np.concatenate(all_latents, axis=0)
labels = np.array(all_labels)

# 2. Run t-SNE
# 'perplexity' controls the balance between local and global structure (try 30-50)
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_jobs=-1)
latents_2d = tsne.fit_transform(latents)

# 3. Visualize
plt.figure(figsize=(10, 8))
scatter = plt.scatter(latents_2d[:, 0], latents_2d[:, 1], c=labels, cmap='viridis', s=5)
plt.colorbar(scatter)
plt.title("VAE Latent Space Visualization via t-SNE")
plt.show()

Input shape before encoder: torch.Size([2048, 3])
Index: 0
Input shape before encoder: torch.Size([2048, 3])
Index: 1
Input shape before encoder: torch.Size([2048, 3])
Index: 2


ValueError: perplexity (30) must be less than n_samples (3)

In [ ]:
#Decoder interpolation

In [14]:
# Optionally, save the generated point clouds to disk
path = pathlib.Path(f"output/{experiment_name}")
path.mkdir(parents=True, exist_ok=True)
generated_pcd_ddim.export(path / f"ddim{DDIM_steps}_{number_of_points}.obj")
# generated_pcd_ddpm.export(f"output/{experiment_name}_ddpm.obj")